In [10]:
# ============================================================
#  Helper Functions Needed for Unified Batch Pipeline
# ============================================================

# -------------------------------
# ACTION MAP (IR → Playwright)
# -------------------------------
ACTION_MAP = {
    "navigate": lambda step: f'        await page.goto("{step["value"]}")',
    "enter_text": lambda step: f'        await page.fill("{step["selector"]}", "{step["value"]}")',
    "click": lambda step: f'        await page.click("{step["selector"]}")',
    "assert_text": lambda step: (
        f'        await expect(page.locator("{step["selector"]}")).to_have_text("{step["value"]}")'
    ),
    "assert_visible": lambda step: (
        f'        await expect(page.locator("{step["selector"]}")).to_be_visible()'
    ),
    "press_enter": lambda step: f'        await page.press("{step["selector"]}", "Enter")'
}

# -------------------------------
# IR → Mapped IR
# -------------------------------
def map_ir(ir):
    mapped_steps = []
    for step in ir["steps"]:
        action = step["action"]
        if action not in ACTION_MAP:
            raise ValueError(f"Unsupported action: {action}")
        mapped_steps.append(step)
    return {
        "test_name": ir["test_name"],
        "steps": mapped_steps
    }

def generate_ir(nl_text):
    """
    Parser for your structured test case format:

    Goal: ...
    URL: ...
    Steps:
    1. ...
    2. ...
    Expected Result: ...
    """

    lines = [line.strip() for line in nl_text.split("\n") if line.strip()]

    # First line becomes test_name
    test_name = lines[0].replace("Goal:", "").strip().lower().replace(" ", "_")

    steps = []
    in_steps = False

    for line in lines:
        lower = line.lower()

        # URL → navigate
        if lower.startswith("url:"):
            url = line.split(":", 1)[1].strip()
            steps.append({"action": "navigate", "value": url})
            continue

        # Steps: → start reading numbered steps
        if lower.startswith("steps"):
            in_steps = True
            continue

        # Expected Result → stop reading steps
        if lower.startswith("expected"):
            in_steps = False
            continue

        # Numbered steps
        if in_steps and line[0].isdigit():
            step_text = line.split(".", 1)[1].strip()
            lower_step = step_text.lower()

            # Log in successfully → shortcut
            if "log in successfully" in lower_step:
                steps.append({"action": "enter_text", "selector": "#username", "value": "demo"})
                steps.append({"action": "enter_text", "selector": "#password", "value": "password123"})
                steps.append({"action": "click", "selector": "#login-button"})
                continue

            # Verify X is visible
            if "verify" in lower_step and "visible" in lower_step:
                selector = step_text.split("visible")[0].replace("Verify", "").strip().strip('"')
                steps.append({"action": "assert_visible", "selector": selector})
                continue

            # Verify browser navigates to /something
            if "navigates to" in lower_step:
                url = step_text.split("to")[-1].strip()
                steps.append({"action": "navigate", "value": url})
                continue

            # Verify text appears
            if "verify" in lower_step and "appears" in lower_step:
                value = step_text.split("appears")[0].replace("Verify", "").strip().strip('"')
                steps.append({"action": "assert_text", "selector": "body", "value": value})
                continue

            # Click
            if lower_step.startswith("click"):
                selector = step_text.split("click")[-1].strip().strip('"')
                steps.append({"action": "click", "selector": selector})
                continue

            # Navigate back
            if "navigate back" in lower_step:
                steps.append({"action": "navigate", "value": "/dashboard"})
                continue

            # Fallback: skip
            print(f"[WARN] Unhandled step: {step_text}")

    return {
        "test_name": test_name,
        "steps": steps
    }

# -------------------------------
# Mapped IR → Playwright Test Code
# -------------------------------
def generate_test_code(ir):
    test_name = ir["test_name"]

    header = [
        "from playwright.async_api import async_playwright, expect",
        "import asyncio",
        "",
        f"async def test_{test_name}():",
        "    async with async_playwright() as p:",
        "        browser = await p.chromium.launch(headless=False)",
        "        page = await browser.new_page()",
        ""
    ]

    body = []
    for step in ir["steps"]:
        action = step["action"]
        if action in ACTION_MAP:
            body.append(ACTION_MAP[action](step))
        else:
            body.append(f"        # Unsupported action: {action}")

    footer = [
        "",
        "        await browser.close()",
        "",
        "if __name__ == '__main__':",
        f"    asyncio.run(test_{test_name}())"
    ]

    return "\n".join(header + body + footer)


In [11]:
# ============================================================
#  Unified Batch Pipeline Notebook
#  NL → IR → Mapped IR → Playwright → Execution → Analysis
# ============================================================

from pathlib import Path
import json, subprocess, datetime

# ------------------------------------------------------------
# 1. Load all .txt test cases
# ------------------------------------------------------------
test_case_dir = Path("../sample-data/test_cases")
test_case_files = sorted(test_case_dir.glob("*.txt"))

print(f"Found {len(test_case_files)} test case(s).")

# ------------------------------------------------------------
# 2. Generate IR for each test case
# ------------------------------------------------------------
ir_output_dir = Path("../sample-data/ir_examples")
ir_output_dir.mkdir(parents=True, exist_ok=True)

generated_ir_files = []

for test_file in test_case_files:
    with open(test_file, "r") as f:
        nl_test = f.read()

    ir = generate_ir(nl_test)

    ir_filename = f"{ir['test_name']}.json"
    ir_path = ir_output_dir / ir_filename

    with open(ir_path, "w") as f:
        json.dump(ir, f, indent=2)

    generated_ir_files.append(ir_path)

print("Generated IR files:")
for p in generated_ir_files:
    print(" -", p)

# ------------------------------------------------------------
# 3. Map IR → Mapped IR (batch)
# ------------------------------------------------------------
mapped_output_dir = Path("../sample-data/ir_examples_mapped")
mapped_output_dir.mkdir(parents=True, exist_ok=True)

mapped_ir_files = []

for ir_path in generated_ir_files:
    with open(ir_path, "r") as f:
        ir = json.load(f)

    mapped_ir = map_ir(ir)

    mapped_filename = f"{mapped_ir['test_name']}_mapped.json"
    mapped_path = mapped_output_dir / mapped_filename

    with open(mapped_path, "w") as f:
        json.dump(mapped_ir, f, indent=2)

    mapped_ir_files.append(mapped_path)

print("\nGenerated mapped IR files:")
for p in mapped_ir_files:
    print(" -", p)

# ------------------------------------------------------------
# 4. Generate Playwright test code for each mapped IR
# ------------------------------------------------------------
generated_tests_dir = Path("../generated-tests")
generated_tests_dir.mkdir(parents=True, exist_ok=True)

generated_test_files = []

for mapped_path in mapped_ir_files:
    with open(mapped_path, "r") as f:
        mapped_ir = json.load(f)

    test_code = generate_test_code(mapped_ir)

    test_filename = f"{mapped_ir['test_name']}_test.py"
    test_path = generated_tests_dir / test_filename

    with open(test_path, "w") as f:
        f.write(test_code)

    generated_test_files.append(test_path)

print("\nGenerated Playwright test files:")
for p in generated_test_files:
    print(" -", p)

# ------------------------------------------------------------
# 5. Execute all generated tests (batch)
# ------------------------------------------------------------
artifacts_root = Path("../artifacts/flask_app_run")
artifacts_root.mkdir(parents=True, exist_ok=True)

run_results = []

for test_path in generated_test_files:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    test_name = test_path.stem.replace("_test", "")
    run_dir = artifacts_root / f"run_{test_name}_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        ["python", str(test_path)],
        capture_output=True,
        text=True
    )

    summary = {
        "test_file": str(test_path),
        "return_code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "artifacts_dir": str(run_dir)
    }

    with open(run_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    run_results.append(summary)

print("\nExecution results:")
for r in run_results:
    print(" -", r["test_file"], "→", "PASS" if r["return_code"] == 0 else "FAIL")

# ------------------------------------------------------------
# 6. Batch summary report
# ------------------------------------------------------------
print("\n==============================")
print("      Batch Test Summary      ")
print("==============================\n")

for result in run_results:
    status = "PASS" if result["return_code"] == 0 else "FAIL"
    print(f"Test: {result['test_file']}")
    print(f"Result: {status}")
    print(f"Artifacts: {result['artifacts_dir']}\n")


Found 5 test case(s).
[WARN] Unhandled step: Navigate to the home page.
[WARN] Unhandled step: Verify the page loads successfully.
[WARN] Unhandled step: Navigate to the login page.
[WARN] Unhandled step: Enter username "wrong".
[WARN] Unhandled step: Enter password "wrong".
[WARN] Unhandled step: Verify the user remains on /login.
[WARN] Unhandled step: Navigate to the login page.
[WARN] Unhandled step: Enter username "demo".
[WARN] Unhandled step: Enter password "password123".
Generated IR files:
 - ../sample-data/ir_examples/test_name:_dashboard_navigation_test.json
 - ../sample-data/ir_examples/test_name:_home_page_test.json
 - ../sample-data/ir_examples/test_name:_login_failure_test.json
 - ../sample-data/ir_examples/test_name:_login_success_test.json
 - ../sample-data/ir_examples/test_name:_logout_flow_test.json

Generated mapped IR files:
 - ../sample-data/ir_examples_mapped/test_name:_dashboard_navigation_test_mapped.json
 - ../sample-data/ir_examples_mapped/test_name:_home_pag